In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.environ['DATASET_25'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'
os.environ['DATASET_15'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'

# verifikasi file benar-benar ada SEBELUM lanjut
print(os.path.exists(os.environ['DATASET_25']))
print(os.path.exists(os.environ['DATASET_15']))

sys.path.append('/content')

Mounted at /content/drive
True
True


In [3]:
sys.path.append('/content/drive/MyDrive')

In [4]:
from common import load_dataset, FEATURES, CLASS_NAMES, SPEED_RANGE

In [5]:
"""
t10_feature_coupling.py  —  T10 (review item M5)

Extends the 25 m/s feature-label coupling analysis (paper Table 9) to the
15 m/s regime so the regime-dependence argument is symmetric. For each raw
feature it reports Pearson r, Spearman rho, and normalized mutual information
MI/H(Y), plus the per-class mean of relative speed (the categorical-looking
separation the paper relies on at 25 m/s).

NEEDS THE DATASET. Set DATASET_PATHS in common.py first, or:
    DATASET_25=/path/Dataset_1.csv DATASET_15=/path/Dataset_2.csv \
        python t10_feature_coupling.py
"""
import os, json
import numpy as np
from scipy.stats import pearsonr, spearmanr, entropy
from sklearn.feature_selection import mutual_info_classif
from common import load_dataset, FEATURES, CLASS_NAMES, SPEED_RANGE

In [6]:
def normalized_mi(X_col, y, seed=0):
    """MI(feature; label) / H(label), in percent. X_col is a 1-D raw feature."""
    mi = mutual_info_classif(X_col.reshape(-1, 1), y,
                             discrete_features=False, random_state=seed)[0]
    # H(Y) in nats (mutual_info_classif also returns nats)
    _, counts = np.unique(y, return_counts=True)
    Hy = entropy(counts)  # natural log
    return 100.0 * mi / Hy if Hy > 0 else 0.0

In [7]:
def coupling_for_speed(target_speed):
    df = load_dataset(target_speed)
    y = df['label'].values
    rows = []
    for feat in FEATURES:
        x = df[feat].values.astype(float)
        pear = pearsonr(x, y)[0] if np.std(x) > 0 else 0.0
        spear = spearmanr(x, y)[0] if np.std(x) > 0 else 0.0
        mi = normalized_mi(x, y)
        rows.append({'feature': feat,
                     'pearson_r': round(float(pear), 3),
                     'spearman_rho': round(float(spear), 3),
                     'mi_over_Hy_pct': round(float(mi), 1)})
    # per-class relative-speed means (the separation visualised in Fig. 6)
    rs_by_class = {}
    for c in range(3):
        rs_by_class[CLASS_NAMES[c]] = round(
            float(df.loc[df['label'] == c, 'Relative_Speed'].mean()), 2)
    return {'speed': target_speed, 'n_samples': int(len(df)),
            'features': rows, 'relative_speed_mean_by_class': rs_by_class}

In [8]:
def main(outdir='outputs_coupling'):
    os.makedirs(outdir, exist_ok=True)
    results = [coupling_for_speed(25), coupling_for_speed(15)]

    for res in results:
        print('\n' + '=' * 78)
        print(f'  T10 — Feature-label coupling @ {res["speed"]} m/s '
              f'(n={res["n_samples"]} raw samples)')
        print('=' * 78)
        print(f'{"Feature":<16s}{"Pearson r":>12s}{"Spearman":>12s}{"MI/H(Y) %":>12s}')
        print('-' * 78)
        for r in res['features']:
            print(f'{r["feature"]:<16s}{r["pearson_r"]:>12.3f}'
                  f'{r["spearman_rho"]:>12.3f}{r["mi_over_Hy_pct"]:>12.1f}')
        print(f'  Relative-speed mean by class: {res["relative_speed_mean_by_class"]}')

    with open(os.path.join(outdir, 't10_coupling.json'), 'w') as f:
        json.dump(results, f, indent=2)

    # side-by-side LaTeX
    lines = [r'\begin{table}[H]',
             r'\caption{Feature--label coupling at both regimes: Pearson, Spearman, '
             r'and normalized mutual information.}',
             r'\centering', r'\begin{tabular}{llrrr}', r'\toprule',
             r'Regime & Feature & Pearson $r$ & Spearman $\rho$ & MI/H(Y) (\%) \\',
             r'\midrule']
    for res in results:
        for r in res['features']:
            lines.append(f'{res["speed"]} m/s & {r["feature"]} & {r["pearson_r"]:+.3f} '
                         f'& {r["spearman_rho"]:+.3f} & {r["mi_over_Hy_pct"]:.1f} \\\\')
        lines.append(r'\midrule')
    lines[-1] = r'\bottomrule'
    lines += [r'\end{tabular}', r'\end{table}']
    with open(os.path.join(outdir, 't10_table.tex'), 'w') as f:
        f.write('\n'.join(lines))
    print(f'\nSaved to {outdir}/')

In [9]:
if __name__ == '__main__':
    main()


  T10 — Feature-label coupling @ 25 m/s (n=1195 raw samples)
Feature            Pearson r    Spearman   MI/H(Y) %
------------------------------------------------------------------------------
SNR                    0.279       0.089        74.7
RSSI                  -0.071      -0.338        76.1
PDR                    0.045       0.089        69.9
Speed                  0.012      -0.003         0.0
Relative_Speed        -0.552      -0.642        47.8
  Relative-speed mean by class: {'Interference': 25.34, 'Reactive': 14.64, 'Constant': 15.35}

  T10 — Feature-label coupling @ 15 m/s (n=1172 raw samples)
Feature            Pearson r    Spearman   MI/H(Y) %
------------------------------------------------------------------------------
SNR                   -0.048      -0.302        55.9
RSSI                  -0.334      -0.559        46.8
PDR                   -0.173      -0.302        45.1
Speed                 -0.014      -0.019         0.0
Relative_Speed        -0.576      -0.663 